# 06 — Ingestion pipeline (end-to-end)

Drop a PDF (with embedded images, ideally) at `notebooks/sample.pdf`.
Then run all cells.

What happens:
1. PDF → Blob
2. Document Intelligence → text + tables
3. PyMuPDF → embedded images → Blob → GPT-4o vision descriptions
4. Smart-chunk + embed
5. Index in AI Search
6. Update Cosmos document metadata

In [1]:
import sys, os, pathlib
sys.path.insert(0, os.path.abspath('..'))
from src.blob_client import BlobService
from src.search_client import SearchService
from src.doc_intelligence import DocIntelService
from src.openai_client import OpenAIService
from src.cosmos_client import create_state_service
from src.ingestion import IngestionPipeline
from src.models import DocumentMeta

blob = BlobService(); blob.ensure_container()
search = SearchService(); search.create_or_update_index()
# CosmosService when configured, else LocalStateService (file-backed JSON).
cosmos = create_state_service(); cosmos.ensure_containers()
print('State backend:', type(cosmos).__name__)
doc_intel = DocIntelService()
openai_svc = OpenAIService()
pipeline = IngestionPipeline(blob, doc_intel, openai_svc, search, cosmos)

INFO: Incomplete environment configuration for EnvironmentCredential. These variables are set: AZURE_TENANT_ID
INFO: ManagedIdentityCredential will use IMDS
INFO: Request URL: 'https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input?restype=REDACTED&sp=REDACTED&st=REDACTED&se=REDACTED&spr=REDACTED&sv=REDACTED&sr=REDACTED&sig=REDACTED'
Request method: 'PUT'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': 'a6ff50ca-4878-11f1-b561-df1619e59047'
No body was attached to the request
INFO: Response status: 403
Response headers:
    'Content-Length': '246'
    'Content-Type': 'application/xml'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': '1eb6cccd-001e-00a5-1085-dc0387000000'
    'x-ms-client-request-id': 'a6ff50ca-4878-11f1-b561-df1619e59047'
    'x-ms-version': 'REDAC

State backend: LocalStateService


In [2]:
PDF = pathlib.Path('Multi_Agent_Research_System_Architecture.pdf')
assert PDF.exists()
doc = DocumentMeta(user_id='nb-user', filename=PDF.name, blob_name=f'nb-user/{PDF.name}')
blob.upload(doc.blob_name, PDF.read_bytes(), content_type='application/pdf')
cosmos.save_document(doc)
print('Doc id:', doc.id)

INFO: Request URL: 'https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/nb-user/Multi_Agent_Research_System_Architecture.pdf?sp=REDACTED&st=REDACTED&se=REDACTED&spr=REDACTED&sv=REDACTED&sr=REDACTED&sig=REDACTED'
Request method: 'PUT'
Request headers:
    'Content-Length': '360829'
    'x-ms-blob-type': 'REDACTED'
    'x-ms-blob-content-type': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Content-Type': 'application/octet-stream'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': 'e277dc97-4878-11f1-a667-df1619e59047'
A body is sent with the request
INFO: Response status: 201
Response headers:
    'Content-Length': '0'
    'Content-MD5': 'REDACTED'
    'Last-Modified': 'Tue, 05 May 2026 11:52:14 GMT'
    'ETag': '"0x8DEAA9CC007C046"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': '1eb7ba6a-001e-00a5-1985-dc038

Doc id: 74af332b-7978-439b-b3f6-6839f78e2f2e


In [3]:
doc = pipeline.process_pdf(doc)
print('Status:', doc.status)
print('Pages :', doc.total_pages)
print('Chunks:', doc.total_chunks)
print('Imgs  :', doc.total_images)
print('Tables:', doc.total_tables)

INFO: Ingesting doc_id=74af332b-7978-439b-b3f6-6839f78e2f2e blob=nb-user/Multi_Agent_Research_System_Architecture.pdf
INFO: Request URL: 'https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/nb-user/Multi_Agent_Research_System_Architecture.pdf?sp=REDACTED&st=REDACTED&se=REDACTED&spr=REDACTED&sv=REDACTED&sr=REDACTED&sig=REDACTED'
Request method: 'GET'
Request headers:
    'x-ms-range': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': 'fc3d1f78-4878-11f1-a358-df1619e59047'
No body was attached to the request
INFO: Response status: 206
Response headers:
    'Content-Length': '360829'
    'Content-Type': 'application/pdf'
    'Content-Range': 'REDACTED'
    'Last-Modified': 'Tue, 05 May 2026 11:52:14 GMT'
    'Accept-Ranges': 'REDACTED'
    'ETag': '"0x8DEAA9CC007C046"'
    'Server': 'Windows-Azure-Blob/1.0 

Status: indexed
Pages : 27
Chunks: 56
Imgs  : 1
Tables: 24


In [4]:
doc

DocumentMeta(id='74af332b-7978-439b-b3f6-6839f78e2f2e', user_id='nb-user', filename='Multi_Agent_Research_System_Architecture.pdf', blob_name='nb-user/Multi_Agent_Research_System_Architecture.pdf', container=None, content_type='application/pdf', size_bytes=0, status='indexed', total_pages=27, total_chunks=56, total_images=1, total_tables=24, error=None, stages=[StageEvent(name='download', status='done', started_at='2026-05-05T11:53:09.017561+00:00', finished_at='2026-05-05T11:53:10.419682+00:00', detail='360829 bytes'), StageEvent(name='extract_text', status='done', started_at='2026-05-05T11:53:10.429771+00:00', finished_at='2026-05-05T11:53:23.167510+00:00', detail='27 pages, 24 tables'), StageEvent(name='extract_images', status='done', started_at='2026-05-05T11:53:23.180335+00:00', finished_at='2026-05-05T11:53:34.573896+00:00', detail='1 images'), StageEvent(name='chunk', status='done', started_at='2026-05-05T11:53:34.588532+00:00', finished_at='2026-05-05T11:53:34.598196+00:00', de